# CCTV Safety — PPE Violation Detection Training
Train YOLOv8 on the Construction Site Safety dataset using a free Colab GPU.

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

In [ ]:
# 1. Verify GPU is available
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU found — change runtime type to T4 GPU')

In [ ]:
# 2. Install dependencies
!pip install ultralytics roboflow --quiet

In [ ]:
# 3. Mount Google Drive to save the trained model persistently
from google.colab import drive
drive.mount('/content/drive')

import os
SAVE_DIR = '/content/drive/MyDrive/cctv-safety-training'
os.makedirs(SAVE_DIR, exist_ok=True)
print(f'Model will be saved to: {SAVE_DIR}')

In [ ]:
# 4. Download dataset from Roboflow
from roboflow import Roboflow

API_KEY = 'AlePU7xXgxgnsQkcin0b'  # your Roboflow private API key

rf = Roboflow(api_key=API_KEY)
project = rf.workspace('roboflow-universe-projects').project('construction-site-safety')
version = project.version(30)
dataset = version.download('yolov8', location='/content/dataset')

print('Dataset path:', dataset.location)
print('Data YAML:',    dataset.location + '/data.yaml')

In [ ]:
# 5. Inspect the data.yaml (verify class names)
import yaml
with open(dataset.location + '/data.yaml') as f:
    data_cfg = yaml.safe_load(f)

print('Classes:', data_cfg['names'])
print('Num classes:', data_cfg['nc'])

In [ ]:
# 6. Train YOLOv8n  (nano = fast, good for edge/CCTV deployment)
#    Swap 'yolov8n.pt' → 'yolov8s.pt' for slightly better accuracy at the cost of speed.
from ultralytics import YOLO

model = YOLO('yolov8n.pt')  # downloads pretrained weights automatically

results = model.train(
    data=dataset.location + '/data.yaml',
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,           # GPU
    project=SAVE_DIR,
    name='ppe_run',
    patience=10,        # early-stop if no improvement for 10 epochs
    save=True,
    plots=True,
)

print('Training complete!')

In [ ]:
# 7. Evaluate on validation set
best_weights = f'{SAVE_DIR}/ppe_run/weights/best.pt'
model_best = YOLO(best_weights)
metrics = model_best.val(data=dataset.location + '/data.yaml', device=0)
print('mAP50:', metrics.box.map50)
print('mAP50-95:', metrics.box.map)

In [ ]:
# 8. Download best.pt to your local machine
from google.colab import files
files.download(best_weights)
print('Downloaded best.pt — place it at  models/ppe.pt  in your local project.')

In [ ]:
# Optional: quick inference test on a val image
import glob
val_images = glob.glob(dataset.location + '/valid/images/*.jpg')[:1]
if val_images:
    result = model_best.predict(val_images[0], conf=0.5, save=True)
    from IPython.display import Image
    display(Image(result[0].save_dir + '/' + val_images[0].split('/')[-1]))